# 🥚 用 NIR 光譜 + PLS 迴歸預測雞蛋儲存天數
### Google Colab 實作教材（Python）

本notebook帶你一步步用**近紅外光譜（NIR, 740–1070 nm）**搭配**偏最小平方迴歸（PLS Regression）**，
預測雞蛋在室溫下已經放了幾天，重現論文的核心分析。

**對應論文：** Coronel-Reyes, J. et al. (2018). *Determination of egg storage time at room temperature using a low-cost NIR spectrometer and machine learning techniques.* Computers and Electronics in Agriculture, 145, 1–10.

**資料集：** SCiO 手持式光譜儀掃描 30 顆褐殼蛋，連續 22 天（第 0–21 天）每天量一次，共 **660 筆**，每筆 **331 個波長**。資料來源：[Mendeley Data](https://data.mendeley.com/datasets/6hn67h2trb/2)。

> **學習目標**：讀懂光譜資料 → 前處理（SNV / Savitzky–Golay 微分）→ 用交叉驗證選 PLS 潛在變數 → 評估模型 → 找出關鍵波長 → 認識「依樣本分組驗證」的重要性。

---
💡 *圖表座標軸用英文是為了避免 Colab 預設無中文字型而顯示方框；所有解說都在中文文字格。*

## 步驟 0：環境與套件
Colab 已內建 numpy / pandas / scipy / scikit-learn / matplotlib，直接 import 即可。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict, KFold, GroupKFold, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA

RNG = 42   # 固定亂數種子，結果可重現
plt.rcParams['figure.dpi'] = 110
print('套件載入完成 ✓')

## 步驟 1：載入資料
請先到 [Mendeley](https://data.mendeley.com/datasets/6hn67h2trb/2) 下載（或用老師提供的）`dataset_egg_storage.csv`。執行下面這格時，若在 Colab 會跳出上傳視窗，選擇該檔案即可。

In [ ]:
import os
CSV = 'dataset_egg_storage.csv'
if not os.path.exists(CSV):
    try:
        from google.colab import files
        print('請上傳 dataset_egg_storage.csv ...')
        up = files.upload()
        CSV = list(up.keys())[0]
    except Exception:
        raise FileNotFoundError('找不到 CSV，請將 dataset_egg_storage.csv 放在同一資料夾')

df = pd.read_csv(CSV)
print('資料維度 (樣本數, 欄位數):', df.shape)
df.iloc[:5, :6]   # 看前 5 筆、前 6 欄

### 整理出 X（光譜）、y（天數）、groups（蛋編號）
- `storage_days`：**目標 y**，雞蛋已儲存的天數（0–21）。
- `sample`：**蛋的編號**（1–30）。同一顆蛋會出現 22 次（每天一筆），這個欄位稍後做「分組驗證」會用到。
- 其餘 `Spectra_740 … Spectra_1070`：**光譜特徵 X**，每個波長的反射率。

In [ ]:
Xcols = [c for c in df.columns if c.startswith('Spectra_')]
wl = np.array([int(c.split('_')[1]) for c in Xcols])   # 波長 740..1070
X = df[Xcols].values.astype(float)
y = df['storage_days'].values.astype(float)
groups = df['sample'].values                            # 蛋編號

print(f'X: {X.shape} | 波長範圍 {wl.min()}–{wl.max()} nm | {len(wl)} 個點')
print(f'y: 天數 {int(y.min())}–{int(y.max())} | 蛋數 {len(np.unique(groups))} 顆 | 每天 {np.sum(y==0)} 顆')

## 步驟 2：探索資料 — 平均光譜如何隨天數改變？
把第 0、1、3、7、14、21 天的**平均光譜**畫出來。你會看到：**儲存越久，整體反射率越高**，尤其在 ~890 nm（C–H）、~960 nm（O–H）、~1020 nm（N–H）這些和蛋殼/蛋白化學鍵相關的位置變化明顯。這正是 PLS 能「讀出」天數的物理基礎。

In [ ]:
day_groups = [0, 1, 3, 7, 14, 21]
colors = plt.cm.viridis(np.linspace(0, 0.95, len(day_groups)))
plt.figure(figsize=(8, 5))
for c, d in zip(colors, day_groups):
    plt.plot(wl, X[y == d].mean(axis=0), color=c, lw=2, label=f'{d} d')
for x0, t in [(890,'C-H'), (960,'O-H'), (1020,'N-H')]:
    plt.axvline(x0, color='gray', ls=':', lw=1); plt.text(x0, plt.ylim()[1], t, ha='center', va='bottom', fontsize=8)
plt.xlabel('Wavelength (nm)'); plt.ylabel('Mean reflectance')
plt.title('Mean NIR spectra by storage day'); plt.legend(title='storage day', ncol=2)
plt.tight_layout(); plt.show()

## 步驟 3：光譜前處理（Preprocessing）
原始光譜含有**散射、基線漂移**等與化學無關的干擾。常見前處理：

| 方法 | 作用 |
|---|---|
| **SNV**（Standard Normal Variate）| 每條光譜各自標準化，去除散射造成的乘性/加性偏移 |
| **Savitzky–Golay 平滑** | 在小窗內以多項式擬合，降噪 |
| **一階微分** | 去除固定基線偏移、突顯斜率變化 |
| **二階微分** | 進一步去除線性基線、突顯吸收峰（但會放大雜訊） |

下面定義這些函式，並比較處理後的平均光譜。

In [ ]:
def snv(s):
    return (s - s.mean(axis=1, keepdims=True)) / s.std(axis=1, keepdims=True)

def sg(s, deriv=0, window=15, poly=2):
    return savgol_filter(s, window_length=window, polyorder=poly, deriv=deriv, axis=1)

PRE = {
    'Raw':            lambda s: s,
    'SNV':            lambda s: snv(s),
    'SavGol 1st der': lambda s: sg(s, deriv=1),
    'SavGol 2nd der': lambda s: sg(s, deriv=2),
}

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (name, fn) in zip(axes.ravel(), PRE.items()):
    Xp = fn(X)
    for c, d in zip(colors, day_groups):
        ax.plot(wl, Xp[y == d].mean(axis=0), color=c, lw=1.3)
    ax.set_title(name); ax.set_xlabel('Wavelength (nm)')
fig.suptitle('Effect of preprocessing on mean spectra'); fig.tight_layout(); plt.show()

## 步驟 4：PLS 是什麼？為什麼適合光譜？
光譜有 **331 個波長（特徵）**，但相鄰波長高度相關（共線性），且特徵數可能比樣本數還多。普通最小平方迴歸在這種情況會崩潰。

**PLS（偏最小平方）** 的作法：找出少數幾個**潛在變數（latent variables, LV）**，這些 LV 是原始波長的線性組合，且被挑選成「**和 y（天數）最相關**」的方向。於是把 331 維壓縮成 ~10 維，同時保留對預測天數最有用的資訊。

關鍵超參數就是 **要用幾個 LV**：太少 → 欠擬合；太多 → 過擬合。我們用**交叉驗證**來選。

## 步驟 5：用 10-fold 交叉驗證選潛在變數數量
對每種前處理、每個 LV 數量，算出**每一折的 RMSE**，再取平均得到 **RMSECV**（交叉驗證均方根誤差，單位：天）。
我們同時畫出 **±1 標準誤** 的誤差棒。

In [ ]:
def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

def cv_rmse_per_fold(Xp, y, k, cv):
    sc = []
    for tr, te in cv.split(Xp, y):
        m = PLSRegression(n_components=k, scale=True).fit(Xp[tr], y[tr])
        sc.append(rmse(y[te], m.predict(Xp[te]).ravel()))
    return np.array(sc)

MAX_LV = 20
kf = KFold(n_splits=10, shuffle=True, random_state=RNG)
plt.figure(figsize=(8.5, 5))
summary = {}
for (name, fn), col in zip(PRE.items(), ['#888','#1f77b4','#d62728','#2ca02c']):
    Xp = fn(X)
    means, ses = [], []
    for k in range(1, MAX_LV+1):
        s = cv_rmse_per_fold(Xp, y, k, kf)
        means.append(s.mean()); ses.append(s.std(ddof=1)/np.sqrt(len(s)))
    means, ses = np.array(means), np.array(ses)
    k_min = means.argmin() + 1
    # 一倍標準誤法則：選 RMSECV 落在 (最小值 + 1SE) 內、最小的 LV (最精簡)
    k_1se = int(np.argmax(means <= means[k_min-1] + ses[k_min-1]) + 1)
    summary[name] = (k_1se, means[k_1se-1])
    plt.errorbar(range(1, MAX_LV+1), means, yerr=ses, fmt='o-', color=col, capsize=2, ms=3,
                 label=f'{name}: 1-SE -> {k_1se} LV ({means[k_1se-1]:.2f} d)')
    plt.plot(k_1se, means[k_1se-1], '*', color=col, ms=15)
plt.xlabel('n_components (LV)'); plt.ylabel('RMSECV (days)')
plt.title('RMSECV vs n_components (* = 1-SE rule)'); plt.legend(fontsize=8); plt.xticks(range(1, MAX_LV+1, 2))
plt.tight_layout(); plt.show()
summary

### 為什麼用「一倍標準誤法則」？
RMSECV 曲線在 LV 很多時往往還在緩緩下降，直接取最低點常會選到 ~19–20 個 LV（**過擬合風險**）。
**一倍標準誤法則（1-SE rule）**：在「最低 RMSECV + 1 個標準誤」的範圍內，選**最少的 LV**。
這樣得到的模型更簡單、更穩健，且和最佳值在統計上沒有顯著差異。

結果：**SavGol 2nd derivative 只用 ~12 個 LV** 就達到最低 RMSECV，比其他前處理（需 ~18–19 LV）更精簡，因此我們選它。

## 步驟 6：訓練最終 PLS 模型並評估
用選定的前處理與 LV 數量，做兩件事：
1. **保留測試集**（20%）：模擬「全新樣本」。
2. **10-fold 交叉驗證**：用全部資料估計穩定的 R² 與 RMSECV。

In [ ]:
BEST_PRE = 'SavGol 2nd der'
BEST_K   = int(summary[BEST_PRE][0])
Xp = PRE[BEST_PRE](X)
print(f'選定前處理 = {BEST_PRE}, 潛在變數 = {BEST_K}')

# (1) 保留測試集
Xtr, Xte, ytr, yte = train_test_split(Xp, y, test_size=0.2, random_state=RNG)
model = PLSRegression(n_components=BEST_K, scale=True).fit(Xtr, ytr)
yhat_te = model.predict(Xte).ravel()
print(f'測試集  R2 = {r2_score(yte, yhat_te):.3f}   RMSE = {rmse(yte, yhat_te):.2f} 天')

# (2) 10-fold 交叉驗證（用全部資料）
yhat_cv = cross_val_predict(PLSRegression(n_components=BEST_K, scale=True), Xp, y, cv=kf).ravel()
print(f'隨機CV  R2 = {r2_score(y, yhat_cv):.3f}   RMSECV = {rmse(y, yhat_cv):.2f} 天   MAE = {mean_absolute_error(y, yhat_cv):.2f} 天')

### 畫「預測 vs 實際」散佈圖
理想情況所有點應落在對角線 *y = x* 上。注意擬合線斜率略小於 1 — 這是 PLS 常見的**向均值收縮**（很新鮮和很舊的蛋會被稍微往中間預測）。

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y, yhat_cv, s=20, alpha=0.4, edgecolor='none')
lims = [-1, 22]; plt.plot(lims, lims, 'k--', label='ideal y=x')
z = np.polyfit(y, yhat_cv, 1); plt.plot(lims, np.polyval(z, lims), 'r-', label=f'fit slope={z[0]:.2f}')
plt.xlim(lims); plt.ylim(lims); plt.gca().set_aspect('equal')
plt.xlabel('Actual storage days'); plt.ylabel('Predicted storage days')
plt.title(f'PLS prediction (R2={r2_score(y, yhat_cv):.3f}, RMSECV={rmse(y, yhat_cv):.2f} d)')
plt.legend(loc='upper left'); plt.tight_layout(); plt.show()

## 步驟 7：⚠️ 驗證的陷阱 — 隨機分折 vs 依「蛋」分組
上面的隨機 10-fold 有個隱憂：**同一顆蛋**第 5 天和第 6 天的光譜很像，若一個在訓練集、一個在驗證集，模型可能「**認得這顆蛋**」而非真的學會判斷天數 —— 這叫**資料洩漏**。

更誠實的作法是 **GroupKFold**：把同一顆蛋的所有 22 筆資料**整組**放在同一折，模擬「預測一顆從沒見過的新蛋」。比較兩者就知道模型是否真的可泛化。

In [ ]:
gkf = GroupKFold(n_splits=10)
yhat_grp = cross_val_predict(PLSRegression(n_components=BEST_K, scale=True),
                             Xp, y, cv=gkf, groups=groups).ravel()
print('隨機 10-fold CV     : R2 = %.3f  RMSECV = %.2f 天' % (r2_score(y, yhat_cv),  rmse(y, yhat_cv)))
print('依蛋分組 GroupKFold : R2 = %.3f  RMSECV = %.2f 天' % (r2_score(y, yhat_grp), rmse(y, yhat_grp)))
print('\n兩者差距很小 -> 模型主要學到的是「老化」而非「蛋的身分」，泛化能力良好 ✓')

## 步驟 8：哪些波長最重要？（VIP 與迴歸係數）
**VIP（Variable Importance in Projection）** 衡量每個波長對 PLS 模型的貢獻，一般以 **VIP > 1** 視為重要。把它畫出來，對照化學鍵的吸收位置，可解釋模型「看」了哪裡。

In [ ]:
def vip(model):
    t, w, q = model.x_scores_, model.x_weights_, model.y_loadings_
    p, h = w.shape
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    return np.sqrt(p * ((w / np.linalg.norm(w, axis=0))**2 @ s).ravel() / s.sum())

m_full = PLSRegression(n_components=BEST_K, scale=True).fit(Xp, y)
vips = vip(m_full)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
a1.plot(wl, m_full.coef_.ravel()); a1.axhline(0, color='gray', lw=.8); a1.set_ylabel('PLS coef')
a2.plot(wl, vips, color='#d62728'); a2.axhline(1, color='gray', ls='--', label='VIP=1')
a2.set_ylabel('VIP'); a2.set_xlabel('Wavelength (nm)'); a2.legend()
for x0, t in [(890,'C-H'),(960,'O-H'),(1020,'N-H')]:
    a1.axvline(x0, color='g', ls=':'); a2.axvline(x0, color='g', ls=':')
plt.suptitle('PLS coefficients & VIP'); plt.tight_layout(); plt.show()
print('VIP 最高的 8 個波長 (nm):', sorted(wl[np.argsort(vips)[::-1][:8]].tolist()))

## 步驟 9（選做）：PCA 探索與離群值
PCA 是非監督式降維，可快速看資料結構與是否有離群點。這裡前兩個主成分就解釋了 ~94% 的變異。

In [ ]:
pca = PCA(n_components=2).fit(snv(X)); sc = pca.transform(snv(X))
plt.figure(figsize=(7, 5.5))
p = plt.scatter(sc[:,0], sc[:,1], c=y, cmap='viridis', s=22, alpha=.8)
plt.colorbar(p, label='storage days')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA score plot (SNV)'); plt.tight_layout(); plt.show()

## 🎯 動手練習
1. **改前處理**：把 `BEST_PRE` 換成 `'SNV'` 或 `'Raw'`，重跑步驟 6，比較 RMSECV 差多少。
2. **改 LV 數量**：手動把 `BEST_K` 設成 2、5、19，觀察「預測 vs 實際」圖與過/欠擬合。
3. **窗寬實驗**：改 `sg(..., window=15)` 的 window（如 7、25），看二階微分的雜訊變化。
4. **二元分類**：論文提到室溫下蛋最多放 ~14 天。試著把問題改成「新鮮(≤7天) vs 不新鮮(>7天)」分類，用 `PLS-DA` 或邏輯迴歸做做看。
5. **挑波長**：只用 VIP > 1 的波長重建模型，準確度會變好還是變差？

---
### 小結
我們用 SavGol 二階微分 + 12 個潛在變數的 PLS，在 740–1070 nm 低成本 NIR 光譜上，達到 **R² ≈ 0.82、RMSECV ≈ 2.7 天**，且依蛋分組驗證結果幾乎相同，證明模型可泛化到新蛋。
這與論文結論一致：**低成本手持光譜儀 + 化學計量學，足以非破壞性地估計雞蛋的儲存天數**。

📄 論文：Coronel-Reyes et al. (2018), *Comput. Electron. Agric.* 145, 1–10.　
📊 資料：https://data.mendeley.com/datasets/6hn67h2trb/2